In [ ]:
import os
import sys
import pandas as pd
from IPython.display import display, Markdown
pd.set_option('display.max_rows', 500)
pd.set_option('display.min_rows', 20)

import collections
import csv
from difflib import SequenceMatcher
from math import ceil

#### input data

In [ ]:
registration_form_data_export=pd.read_csv('/Users/daelsaid/scsnl_docs/redcap/registration_form_rework_2023/reg_form_data_export_labels_with_variable__headers.csv',dtype=str,skiprows=[1],encoding='utf-8')
first_order_relatives_orig=pd.read_csv('/Users/daelsaid/scsnl_docs/redcap/registration_form_rework_2023/first_order_relatives.csv',dtype=str,skiprows=[0],encoding='utf-8')


output_filenamne_prefix='scsnl_registrationform_remap'

In [ ]:
first_order_relatives_orig.rename(columns={'participant_psycfam___1':'participant_psycfam___1_Autism','participant_psycfam___2':'participant_psycfam___2_ADHD','participant_psycfam___3':'participant_psycfam___3_Dyslexia/Language disorders','participant_psycfam___4':'participant_psycfam___4_Dyscalculia/math disability','participant_psycfam___5':'participant_psycfam___5_Schizophrenia','participant_psycfam___6':'participant_psycfam___6_Bipolar Disorder','participant_psycfam___7':'participant_psycfam___7_Fragile X syndrome','participant_psycfam___8':'participant_psycfam___8_Epilepsy','participant_psycfam___11':'participant_psycfam___11_Major Depression','participant_psycfam___9':'participant_psycfam___9_Other developmental delay or intellectual disability','participant_psycfam___10':'participant_psycfam___10_Other psychiatric and neurologic disorders','psyc_famrel1':'psyc_famrel1_Autism','psyc_famrel2':'psyc_famrel2_ADHD','psyc_famrel3':'psyc_famrel3_Dyslexia/Language disorders','psyc_famrel4':'psyc_famrel4_Dyscalculia/math disability','psyc_famrel5':'psyc_famrel5_Schizophrenia','psyc_famrel6':'psyc_famrel6_Bipolar Disorder','psyc_famrel7':'psyc_famrel7_Fragile X Syndrome','psyc_famrel8':'psyc_famrel8_Epilepsy','psyc_famrel9':'psyc_famrel9_Other developmental delay or intellectual disability','psyc_famrel10':'psyc_famrel10_Other psychiatric and neurological disorders','psyc_famrel12':'psyc_famrel12_Major Depression','psyc_extra':'psyc_extra_general'},inplace=True)

first_order_relatives=first_order_relatives_orig[['record_id','psyc_famrel1_Autism','participant_psycfam___1_Autism','psyc_famrel2_ADHD','participant_psycfam___2_ADHD','psyc_famrel3_Dyslexia/Language disorders','participant_psycfam___3_Dyslexia/Language disorders','psyc_famrel4_Dyscalculia/math disability','participant_psycfam___4_Dyscalculia/math disability','psyc_famrel5_Schizophrenia','participant_psycfam___5_Schizophrenia','psyc_famrel6_Bipolar Disorder','participant_psycfam___6_Bipolar Disorder','psyc_famrel7_Fragile X Syndrome','participant_psycfam___7_Fragile X syndrome','psyc_famrel8_Epilepsy','participant_psycfam___8_Epilepsy','psyc_famrel9_Other developmental delay or intellectual disability','participant_psycfam___9_Other developmental delay or intellectual disability','psyc_famrel12_Major Depression','participant_psycfam___11_Major Depression','psyc_famrel10_Other psychiatric and neurological disorders','participant_psycfam___10_Other psychiatric and neurologic disorders','psyc_extra_general']]

In [ ]:
print(first_order_relatives.columns)

#### 1:first order relative
#### get unique responses 


This code appears to be selecting columns from a pandas dataframe called "first_order_relatives". Specifically, it is selecting columns with names that contain either "psyc_famrel" or "psyc_extra".

The code then loops through a list of selected relatives and filters the "first_order_relatives" dataframe based on whether each relative has any associated psychiatric disorders or conditions.

The possible options for each relative are added to a list called "list_of_responses", and then a final list called "final_unique_resp" is created by removing duplicates from "list_of_responses".



In [ ]:
# cols_to_update=[x for x in first_order_relatives.columns.tolist() if 'psyc_famrel' in x and 'Schizophrenia' in x or 'Bipolar' in x or 'Other' in x or 'Depression' in x]
cols_to_update=[x for x in first_order_relatives.columns.tolist() if 'psyc_famrel' in x  or 'psyc_extra' in x]
list_of_responses=[]

# for idx,relative in enumerate(['psyc_famrel5_Schizophrenia','psyc_famrel6_Bipolar Disorder','psyc_famrel10_Other psychiatric and neurological disorders','psyc_famrel12_Major Depression']):
for idx,relative in enumerate(['psyc_famrel1_Autism','psyc_famrel2_ADHD','psyc_famrel3_Dyslexia/Language disorders','psyc_famrel4_Dyscalculia/math disability','psyc_famrel5_Schizophrenia','psyc_famrel6_Bipolar Disorder','psyc_famrel7_Fragile X Syndrome','psyc_famrel8_Epilepsy','psyc_famrel9_Other developmental delay or intellectual disability','psyc_famrel10_Other psychiatric and neurological disorders','psyc_famrel12_Major Depression','psyc_extra_general']):
    rel_options=first_order_relatives[(first_order_relatives['participant_psycfam___5_Schizophrenia'] != "0")
                      | (first_order_relatives['participant_psycfam___6_Bipolar Disorder'] != "0")
                      | (first_order_relatives['participant_psycfam___10_Other psychiatric and neurologic disorders'] != "0")
                      | (first_order_relatives['participant_psycfam___11_Major Depression'] != "0")
                      & (first_order_relatives[relative] != "")][relative].str.lower().unique().tolist()    
    rel_options_all=first_order_relatives[(first_order_relatives['participant_psycfam___5_Schizophrenia'] != "0")
                                                    | (first_order_relatives['participant_psycfam___6_Bipolar Disorder'] != "0")
                                                    | (first_order_relatives['participant_psycfam___10_Other psychiatric and neurologic disorders'] != "0")
                                                    | (first_order_relatives['participant_psycfam___1_Autism'] != "0")
                                                    | (first_order_relatives['participant_psycfam___2_ADHD'] != "0")
                                                    | (first_order_relatives['participant_psycfam___3_Dyslexia/Language disorders'] != "0")
                                                    | (first_order_relatives['participant_psycfam___4_Dyscalculia/math disability'] != "0")
                                                    | (first_order_relatives['participant_psycfam___8_Epilepsy'] != "0")
                                                    | (first_order_relatives['participant_psycfam___7_Fragile X syndrome'] != "0")
                                                    | (first_order_relatives['participant_psycfam___9_Other developmental delay or intellectual disability'] != "0")
                                                    | (first_order_relatives['participant_psycfam___11_Major Depression'] != "0")][relative].str.lower().unique().tolist()
    # list_of_responses.append([set(rel_options)])
    for item in set(rel_options_all):
        list_of_responses.append(item)
        
       
# for i in range(0,len(rel_options)):
#         print(unique(rel_options[i]))
final_unique_resp = []
for item in list_of_responses:
    if item not in final_unique_resp:
        final_unique_resp.append(item)

'psyc_famrel5_Schizophrenia  
psyc_famrel6_Bipolar Disorder  	
psyc_famrel10_Other psychiatric and neurological disorders	  
psyc_famrel12_Major Depression	  


relation between your child and the family member with Schizophrenia  
·         Relation between your child and the family member with Bipolar Disorder  
·         Relation between your child and the family member with Other psychiatric and neurological disorders  
·         Relation between your child and the family member with Major Depression  


#### 2. create dictionary of final unique responses based off of columns 
- make dictionary called "final_unique_resp". The dictionary is initially empty.
- loops through the list of columns selected in the previous code block, and for each column it creates a dictionary from the "first_order_relatives" dataframe that maps each unique value in the column to a set of all the unique values in that column, after converting all the values to lowercase.
- If the current column contains either "famrel" or "psyc_extra", the code adds the set of unique values to the "final_unique_resp" dictionary under the current column name as a list.
- Finally, the code prints the keys of the "final_unique_resp" dictionary, which will be the column names for which unique values were added as lists to the dictionary.





In [ ]:
final_unique_resp=dict()

for col in cols_to_update:
    dict_from_df = {cols: set(first_order_relatives[cols].str.lower().unique()) for cols in first_order_relatives.columns}
    if 'famrel' in col or 'psyc_extra' in col:        
        final_unique_resp[col]=[]
        final_unique_resp[col]=dict_from_df[col]
# print dictionary
print(final_unique_resp.keys())


### 3. make empty dict 

In [ ]:
responses_firstorder = {'Parent: mother': [],
                        'Parent: Father': [],
                        'Sibling: Brother': [],
                        'Sibling: Sister': [],
                        'Other Family Member': []
}

### 4. group responses

This code is looping through each key-value pair in the final_unique_resp dictionary, and then it's looping through each value in the dictionary's value (which is a set). 
For each value, it's checking if it contains certain keywords (e.g. "mom", "dad", "sister", etc.), and then appending that value to the corresponding list in the responses_firstorder dictionary.
If the value doesn't match any of the keywords in the if statements, it's being appended to the 'Other Family Member' list in the responses_firstorder dictionary.



In [ ]:
for key in final_unique_resp.keys():
    # print(key)
    # print(responses_firstorder)

    for val in final_unique_resp[key]:
        if 'grandmother' not in str(val) and ('mo' in str(val) or 'mom' in str(val) or 'om' in str(val) or 'arent' in str(val)):
            responses_firstorder['Parent: mother'].append(str(val))
        if 'grandfather' not in str(val) and ('fa' in str(val) or 'dad' in str(val) or 'ather' in str(val) or 'arent' in str(val)):
            responses_firstorder['Parent: Father'].append(str(val))
        if ('son' in str(val) or 'child' in str(val) or 'bro' in str(val) or 'ibling' in str(val)) and ('1/2' not in str(val) and 'half' not in str(val)):
            responses_firstorder['Sibling: Brother'].append(str(val))
        if ('sis' in str(val) or 'daughter' in str(val) or 'child' in str(val) or 'ibling' in str(val) or 'ist' in str(val) or 'ister' in str(val)) and ('1/2' not in str(val) and 'half' not in str(val)):
            responses_firstorder['Sibling: Sister'].append(str(val))
        if 'grandfather' in str(val) or 'grandparent' in str(val) or 'grandmother' in str(val) or '1/2' in str(val) or 'half' in str(val) or 'aunt' in str(val) or 'uncle' in str(val):
            responses_firstorder['Other Family Member'].append(str(val))
        else: 
            responses_firstorder['Other Family Member'].append(str(val))

### 5. Make final dictionary 

This code block is iterating over the dictionary responses_firstorder that was created earlier in the script, and for each key (representing a category of first-order relatives), it is removing any duplicate values in the associated list of responses by converting the list to a set (which automatically removes duplicates), and then back to a list.

The result is that the responses_firstorder dictionary will contain a list of unique responses for each category of first-order relatives (e.g. "Parent: mother", "Parent: father", "Sibling: Brother", etc.)

In [ ]:
for key in responses_firstorder:
    # Convert the list to a set to remove duplicates
    unique_vals = set(responses_firstorder[key])
    # Convert the set back to a list
    responses_firstorder[key] = list(unique_vals)



       'participant_psycfam___5_Schizophrenia',
       'participant_psycfam___6_Bipolar Disorder',
       'psyc_famrel12_Major Depression', 'psyc_extra_general'],
       'psyc_famrel10_Other psychiatric and neurological disorders',

### First order relative responses mapped out to new checkboxes


This code block seems to be checking if a given response for a first-order relative (such as "Parent: mother") is present in the column names containing information about various psychiatric and neurological disorders for that relative.   
If the response is present in any of those columns, then the corresponding cell in the same row but in a new column with a suffix '_'+ relative (such as "participant_psycfam___5_Schizophrenia_Parent: mother") is set to 1.   
This allows the dataset to be updated with information about the psychiatric and neurological history of the participant's first-order relatives.  





In [ ]:
for relative in responses_firstorder.keys():
    for rel_response in responses_firstorder[relative]:
            # Check if the med value is present in the psycmeds_desc column
            is_present_other= first_order_relatives[(first_order_relatives['participant_psycfam___10_Other psychiatric and neurologic disorders'] != "0")]['psyc_famrel10_Other psychiatric and neurological disorders'].str.contains(rel_response,case=False,regex=True).any()
            
            is_present_mdd= first_order_relatives[(first_order_relatives['participant_psycfam___11_Major Depression'] != "0")]['psyc_famrel12_Major Depression'].str.contains(rel_response,case=False,regex=True).any()
            
            is_present_sz= first_order_relatives[(first_order_relatives['participant_psycfam___5_Schizophrenia'] != "0")]['psyc_famrel5_Schizophrenia'].str.contains(rel_response,case=False,regex=True).any()
            
            is_present_bpdx=first_order_relatives[(first_order_relatives['participant_psycfam___6_Bipolar Disorder'] != "0")]['psyc_famrel6_Bipolar Disorder'].str.contains(rel_response,case=False,regex=True).any()
            
            is_present_other_dev= first_order_relatives[(first_order_relatives['participant_psycfam___9_Other developmental delay or intellectual disability'] != "0")]['psyc_famrel9_Other developmental delay or intellectual disability'].str.contains(rel_response,case=False,regex=True).any()
            
            is_present_dyscalcy= first_order_relatives[(first_order_relatives['participant_psycfam___4_Dyscalculia/math disability'] != "0")]['psyc_famrel4_Dyscalculia/math disability'].str.contains(rel_response,case=False,regex=True).any()
            
            is_present_epilepsy= first_order_relatives[(first_order_relatives['participant_psycfam___8_Epilepsy'] != "0")]['psyc_famrel8_Epilepsy'].str.contains(rel_response,case=False,regex=True).any()
            
            is_present_fx= first_order_relatives[(first_order_relatives['participant_psycfam___7_Fragile X syndrome'] != "0")]['psyc_famrel7_Fragile X Syndrome'].str.contains(rel_response,case=False,regex=True).any()
            
            is_present_asd= first_order_relatives[(first_order_relatives['participant_psycfam___1_Autism'] != "0")]['psyc_famrel1_Autism'].str.contains(rel_response,case=False,regex=True).any()
            
            is_present_adhd= first_order_relatives[(first_order_relatives['participant_psycfam___2_ADHD'] != "0")]['psyc_famrel2_ADHD'].str.contains(rel_response,case=False,regex=True).any()
            
            is_present_dys= first_order_relatives[(first_order_relatives['participant_psycfam___3_Dyslexia/Language disorders'] != "0")]['psyc_famrel3_Dyslexia/Language disorders'].str.contains(rel_response,case=False,regex=True).any()
            
            
            if is_present_bpdx:
                first_order_relatives.loc[first_order_relatives['psyc_famrel6_Bipolar Disorder'].str.contains(rel_response).fillna(False), relative+'_psyc_famrel6_bipolar']='1'

            if is_present_mdd:
                first_order_relatives.loc[first_order_relatives['psyc_famrel12_Major Depression'].str.contains(rel_response).fillna(False), relative+'_psyc_famrel11_mdd']='1'
                
            if is_present_other:
                first_order_relatives.loc[first_order_relatives['psyc_famrel10_Other psychiatric and neurological disorders'].str.contains(rel_response).fillna(False), relative+'_psyc_famrel10_other_psych_neuro_dx']='1'

            if is_present_sz:
                first_order_relatives.loc[first_order_relatives['psyc_famrel5_Schizophrenia'].str.contains(rel_response).fillna(False), relative+'_psyc_famrel5_sz']='1'
                
                
            if is_present_dys:            
                first_order_relatives.loc[first_order_relatives['psyc_famrel3_Dyslexia/Language disorders'].str.contains(rel_response).fillna(False), relative+'_psyc_famrel3_dyslexia_language_dx']='1'
                
            if is_present_adhd:
                first_order_relatives.loc[first_order_relatives['psyc_famrel2_ADHD'].str.contains(rel_response).fillna(False), relative+'_psyc_famrel2_adhd']='1'
                
            if is_present_asd:
                first_order_relatives.loc[first_order_relatives['psyc_famrel1_Autism'].str.contains(rel_response).fillna(False), relative+'_psyc_famrel1_asd']='1'
                
            if is_present_fx:
                first_order_relatives.loc[first_order_relatives['psyc_famrel7_Fragile X Syndrome'].str.contains(rel_response).fillna(False), relative+'_psyc_famrel7_fragile_x_syndrome']='1'
                
            if is_present_epilepsy:
                first_order_relatives.loc[first_order_relatives['psyc_famrel8_Epilepsy'].str.contains(rel_response).fillna(False), relative+'_psyc_famrel8_epilepsy']='1'
                
            if is_present_dyscalcy:
                first_order_relatives.loc[first_order_relatives['psyc_famrel4_Dyscalculia/math disability'].str.contains(rel_response).fillna(False), relative+'_psyc_famrel4_math_dyscalculia_dx']='1'
                
            if is_present_other_dev:
                first_order_relatives.loc[first_order_relatives['psyc_famrel9_Other developmental delay or intellectual disability'].str.contains(rel_response).fillna(False), relative+'_psyc_famrel9_other_intel_dis_dev_delay']='1'
                


In [ ]:
first_order_relatives_cols_opposite = {'participant_psycfam___1':'participant_psycfam___1_Autism',
                                       'participant_psycfam___2':'participant_psycfam___2_ADHD',
                                       'participant_psycfam___3':'participant_psycfam___3_Dyslexia/Language disorders',
                                       'participant_psycfam___4':'participant_psycfam___4_Dyscalculia/math disability',
                                       'participant_psycfam___5':'participant_psycfam___5_Schizophrenia',
                                       'participant_psycfam___6':'participant_psycfam___6_Bipolar Disorder',
                                       'participant_psycfam___7':'participant_psycfam___7_Fragile X syndrome',
                                       'participant_psycfam___8':'participant_psycfam___8_Epilepsy',
                                       'participant_psycfam___11':'participant_psycfam___11_Major Depression',
                                       'participant_psycfam___9':'participant_psycfam___9_Other developmental delay or intellectual disability',
                                       'participant_psycfam___10':'participant_psycfam___10_Other psychiatric and neurologic disorders',
                                       'psyc_famrel1':'psyc_famrel1_Autism',
                                       'psyc_famrel2':'psyc_famrel2_ADHD',
                                       'psyc_famrel3':'psyc_famrel3_Dyslexia/Language disorders',
                                       'psyc_famrel4':'psyc_famrel4_Dyscalculia/math disability',
                                       'psyc_famrel5':'psyc_famrel5_Schizophrenia',
                                       'psyc_famrel6':'psyc_famrel6_Bipolar Disorder',
                                       'psyc_famrel7':'psyc_famrel7_Fragile X Syndrome',
                                       'psyc_famrel8':'psyc_famrel8_Epilepsy',
                                       'psyc_famrel9':'psyc_famrel9_Other developmental delay or intellectual disability',
                                       'psyc_famrel10':'psyc_famrel10_Other psychiatric and neurological disorders',
                                       'psyc_famrel12':'psyc_famrel12_Major Depression',
                                       'psyc_extra':'psyc_extra_general'}


first_order_relatives_cols_opposite_switched = {v: k for k, v in first_order_relatives_cols_opposite.items()}

print(first_order_relatives_cols_opposite_switched)
first_order_relatives.rename(columns={'participant_psycfam___1_Autism': 'participant_psycfam___1', 'participant_psycfam___2_ADHD': 'participant_psycfam___2', 'participant_psycfam___3_Dyslexia/Language disorders': 'participant_psycfam___3', 'participant_psycfam___4_Dyscalculia/math disability': 'participant_psycfam___4', 'participant_psycfam___5_Schizophrenia': 'participant_psycfam___5', 'participant_psycfam___6_Bipolar Disorder': 'participant_psycfam___6', 'participant_psycfam___7_Fragile X syndrome': 'participant_psycfam___7', 'participant_psycfam___8_Epilepsy': 'participant_psycfam___8', 'participant_psycfam___11_Major Depression': 'participant_psycfam___11', 'participant_psycfam___9_Other developmental delay or intellectual disability': 'participant_psycfam___9', 'participant_psycfam___10_Other psychiatric and neurologic disorders': 'participant_psycfam___10', 'psyc_famrel1_Autism': 'psyc_famrel1', 'psyc_famrel2_ADHD': 'psyc_famrel2', 'psyc_famrel3_Dyslexia/Language disorders': 'psyc_famrel3', 'psyc_famrel4_Dyscalculia/math disability': 'psyc_famrel4', 'psyc_famrel5_Schizophrenia': 'psyc_famrel5', 'psyc_famrel6_Bipolar Disorder': 'psyc_famrel6', 'psyc_famrel7_Fragile X Syndrome': 'psyc_famrel7', 'psyc_famrel8_Epilepsy': 'psyc_famrel8', 'psyc_famrel9_Other developmental delay or intellectual disability': 'psyc_famrel9', 'psyc_famrel10_Other psychiatric and neurological disorders': 'psyc_famrel10', 'psyc_famrel12_Major Depression': 'psyc_famrel12', 'psyc_extra_general': 'psyc_extra'},inplace=True)


In [ ]:
first_order_relatives.to_csv(output_filename_prefix+'updated_firstorder_relatives_w_dict.csv')

In [ ]:
diagnosis1={
      '1':{'ADHD/ADD':{'add','add ','adhd ', 'adhd', 'attention deficit','add/adhd','ADHD','ADHD/ADD','adhd/add','behavioral/ add'}
     },
    '2': {'Aggression': {'aggression', 'aggressive', 'explosive', 'anger'}
          },
    '3': {'Anxiety (Generalized, Separation, Social, Panic, Attachment)': {'anx', 'anxious', 'anxiety', 'anxiety nos', 'nos anxiety', 'transition anxiety', 'reactive attachment disorder', 'panic disorder', 'social anxiety', 'social anxiety', 'separation anxiety', 'anxiety (gad-nos)'}
          },
    '4': {'ASD/Autism': {'asd/autism', 'asd', 'hf asd', 'high functioning autism', 'asd', 'autism', 'spectrum'}},
    '5': {'Aspergers': {'asperger\'s', 'aspergers', 'asberger', 'aspie'}
          },
    '6': {'Bipolar disorder': {'bipolar disorder', 'bipolar', 'manic', 'bi-polar', 'bpd', 'Bipolar'}
          },
    '7': {'Depression/Major Depression Disorder (MDD)': {'depressive', 'dep', 'depression', 'MDD', 'depress', 'major depression', 'depressive'}
          },
    '8': {'Emotional Disturbance Disorder/Other Mood Disorder': {'mood disorder', 'mood disorder - nos', 'emotional disturbance disorder'}
          },
    '9': {'Obssessive Compulsive Disorder (OCD)': {'ocd', 'obsessive', 'compulsive', 'OCD'}
          },
    '10': {'Oppositional Defiant Disorder (ODD)': {'oppositional defiant disorder (odd)', 'odd', 'oppositional', 'defiant'}
           },
    '11': {'Pervasive Developmental Disorder Not Otherwise Specified (PDD-NOS)': {'pdd-nos', 'PDD'}
           },
    '12': {'Phobia/specific fear (e.g., claustrophobia)': {'claustrophobic', 'phobia/specific fear', 'phobia'}
           },
    '13': {'PTSD/Trauma/Adjustment Disorder': {'trauma/ptsd', 'adjustment disorder', 'ptsd', 'trauma', 'ptsd', 'adjustment'}
           },
    '14': {'Selective Mutism': {'selective mutism', 'mutism'}
           },
    '15': {'Sensory Integration/Processing Disorder': {'sensory processing', 'sensory', 'sensory processing disorder', 'sensory integration', 'sensory disorder'}
           },
    '16': {'Tic/Tourettes': {'tourette', 'tic'}
           },
    '17': {'Trichiyillomania': {'trichiyillomania'}
           },
    '98': {'NA (not psychological; not listed above)': {'other', 'ed', 'dysgraphia', 'developmental', 'communication disorder'}
           }
}


In [ ]:
d = {'1': {'participant_meds_thyroid': {'cytomel', 'synthroid', 'levothyroxine', 'levoxl', 'levothroidwesthroid', 'liothyronine', 'armourthyroid', 'levothroid', 'propanolol', 'inderalla', 'innopranxl', 'hemangeol', 'inderal', 'yto', 'ynth', 'vothy', 'evox', 'roidwe', 'iothy', 'mourt', 'evothr', 'opan', 'ralla', 'nopra', 'geol', 'inde', 'thyroid'}
           },
     '2': {'participant_meds_blood_thinners': {'atripla', 'stribild', 'complera', 'triumeq', 'heparin', 'lovenox', 'fragmin', 'eliquis', 'pradaxa', 'xarelto', 'plavix', 'warfarin', 'coumadin', 'ipla', 'bild', 'compl', 'umeq', 'hepa', 'love', 'agmi', 'iqui', 'adax', 'xare', 'plav', 'warf', 'umad'}
           },
     '3': {'participant_meds_ssri_MDD': {'pexeva', 'prozac', 'celexa', 'lexapro', 'luvox', 'paxil', 'zoloft', 'cymbalta', 'effexor', 'remeron', 'wellbutrin', 'anafranil', 'norpramin', 'pamelor', 'sinequan', 'surmontil', 'minipress', 'citalopram', 'escitalopram', 'fluoxetine', 'paroxetine', 'sertraline', 'sert', 'duloxetine', 'venlafaxine', 'mirtazapine', 'bupropion', 'clomipramine', 'amitriptyline', 'desipramine', 'nortriptyline', 'doxepin', 'trimipramine', 'prazosin', 'pine', 'pam', 'mine', 'pram'}
           },
     '4': {'participant_meds_psychotic': {'risperdal', 'risperidone', 'risp', 'abilify', 'abil', 'fy' 'zyprexa', 'seroquel', 'geodon', 'therazine', 'prolixin', 'haldol', 'moban', 'navane', 'stelazine', 'loxitane', 'riperidone', 'aripirazole', 'olanzapine', 'quetiapine', 'ziprasidone', 'chlorpromazine', 'fluphenazine', 'haloperidol', 'molindone', 'thiothixene', 'trifluoperazine', 'loxapine', 'adasuve', 'ormazine', 'permitil', 'abil', 'ypre', 'roqu', 'eod', 'heraz', 'olix', 'aldo', 'oba', 'ava', 'elaz', 'xit', 'ripe', 'pira', 'lanz', 'etia', 'ipras', 'oma', 'luph', 'operid', 'olin', 'hix', 'iflu', 'loxa', 'dasu', 'rmaz', 'ermi'}
           },
     '5': {'participant_meds_anti_convulsant': {'depakene', 'depakote', 'tegretol', 'lithobid', 'eskalith', 'oxcarbazepine', 'lamotrigine', 'lamot', 'lithium', 'valproicacid', 'divalproex', 'trileptal', 'lamictal', 'carbamazepine', 'pake', 'pako', 'gret', 'thob', 'skal', 'xcarb', 'motri', 'thium', 'oicac/lproe', 'ilep', 'amic'}
           },
     '6': {'participant_meds_attention_add': {'amphetamine', 'methylphenidate', 'nonstimulants', 'adderall', 'focaline', 'strattera', 'intuniv', 'dexedrine', 'dextrostat', 'vyvanse', 'methylin', 'ritalin', 'metadate', 'concerta', 'quillivant', 'daytrana', 'ritalin', 'hetam', 'ylphen', 'nonsti', 'adde', 'foca', 'strat', 'univ', 'dexed', 'dextros', 'yvans', 'hylin', 'rita', 'tadat', 'conc', 'illiv', 'ytran', 'rit'}
           },
     '7': {'participant_meds_anti_anx': {'alprazolam', 'ativan', 'lorazepam', 'xanax', 'clonazepam', 'valium', 'prazol', 'tiva', 'oraze', 'anax', 'onaze', 'alium'}
           },
     '8': {'participant_meds_sleep': {'melatonin', 'ambien', 'eszopiclone', 'ramelteon', 'triazolam', 'zaleplon', 'aolpidem', 'trazadone', 'lunesta', 'halcion', 'ambien', 'rozerem', 'sonata', 'zopic', 'elteo', 'iazo', 'alepl', 'olpi', 'azad', 'unest', 'alcio', 'mbien', 'ozere', 'onata'}
           },
     '9': {'chronic_pain_meds': {'buprenorphine', 'renorp', 'butorphanol', 'torph', 'codeine', 'ein', 'hydrocodone', 'rocod', 'hydromorphone', 'romorp', 'oxymorphone', 'ymorp', 'propxyphene', 'pyxp', 'tapentadol', 'ntado', 'tramadol', 'ramad', 'dilaudid', 'laudi', 'acetaminophen', 'cetami', 'hydrostat', 'ydros', 'exalgo', 'xalg', 'numorphan', 'umorp', 'contanal', 'ontan', 'darvon', 'arvon', 'ultracet', 'ltrac', 'nucynta', 'ucynt', 'buprenex', 'upren', 'butranspatch', 'ransp', 'stadol', 'tadol', 'morphine', 'rphin', 'nalbuphine', 'lbuph', 'oxycodone', 'ycodo', 'pentazocine', 'ntazo', 'diclofenac', 'clofe', 'meloxicam', 'loxic', 'naproxen', 'eproxe', 'gabapentin', 'bapent', 'pregabalin', 'egaba', 'fentanyl', 'ntany', 'meperidine', 'eridi', 'methadone', 'thado', 'levorphanol', 'vorphan', 'levo-dromoran', 'romora', 'demerol', 'emero', 'dolophine', 'olophi', 'astramorph', 'stramo', 'nubain', 'ubai', 'oxycontin', 'xycont', 'talwin', 'alwi', 'cataflam', 'atafla', 'mobic', 'obic', 'aleve', 'leve', 'neurontin', 'uronti', 'lyrica', 'yric', 'duragesic', 'urage', 'methadose', 'ethad', 'voltaren', 'oltar', 'msir', 'cont', 'kadian', 'adian', 'duramorph', 'uramor', 'avinza', 'vinza', 'roxicodone', 'oxicodo', 'oxecta', 'xecta', 'zipsor', 'ipsor', 'naprelan', 'aprela', 'anaproct', 'anaproc', 'naproc', 'gralise', 'alise', 'naprosyn', 'aprosy', 'butranspatch', 'butran', 'ibuprofen', 'advil', 'tylenol', 'vil', 'ylen', 'uprof'}
           },
     '10': {'participant_adhd_stimulants_need_washout': {'aptensio', 'ptens', 'aptensio xr', 'concerta', 'erta', 'certa', 'daytrana', 'ytran', 'focalin', 'focalin xr', 'ocal', 'metadate', 'metadate er', 'tadat', 'methylin', 'hylin', 'quillivant', 'quillichew', 'illiv', 'ritalin', 'ritalin la', 'rita', 'adderall', 'adderall xr', 'add', 'ral', 'adde', 'dexedrine', 'edrine', 'dyanavel', 'dyanevel xr', 'ynave', 'evekeo', 'veke', 'procentra', 'rocen', 'vyvanse', 'vyv', 'amphetamine', 'dextroamphetamine', 'hetam', 'methylphenidate', 'ylphen', 'dexmethylphenidate', 'xmethy', 'lisdexamfetamine', 'lisd'}
            },
     '11': {'participant_adhd_non_stim': {'intuniv', 'univ', 'kapvay', 'apva', 'wellbutrin', 'lbutri', 'viloxazine', 'vilo', 'guanfacine', 'gua', 'fancine', 'uanf', 'clonidine', 'clon', 'buproprion', 'prion'}
            },
     '12': {'participant_adhd_non_stim_nowashout': {'strattera', 'trat', 'atomoxetine', 'atomoxetine hcl', 'mox'}
            },
     '13': {'participant_meds_allergy_asthma': {'asthma', 'albuterol', 'alb', 'nasonex', 'cetrizine', 'cetri', 'zine', 'nasal', 'spray', 'inhaler', 'flonase', 'montelukast', 'singulair', 'buter', 'inh', 'flo', 'mont', 'ulair', 'zyrtec', 'zyr', 'claritin', 'clar', 'zyzal', 'sinus', 'allergies', 'benadryl', 'dryl', 'loratidine', 'qvar'}
            },
     '14': {'antibiotics': {'antibiotics', 'antib', 'biotic', 'azithro', 'mycine', 'vir', 'thromyc'}
            },
     '15': {'participant_meds_skin': {'ecz', 'ecema', 'retinoid', 'retin-a', 'tacrolimus', 'rolim', 'differin', 'diff', 'accutane', 'accu', 'steroid'}
            }
     }


In [ ]:
for key in d:
    for med_group in d[key]:
        for i, med in enumerate(d[key][med_group]):
            # Check if the med value is present in the psycmeds_desc column
            is_present = registration_form_data_export[(registration_form_data_export['participant_psycmeds'] != "no")
                                                       & (registration_form_data_export['psycmeds_desc'] != "")]['psycmeds_desc'].str.contains(med, case=False, regex=True).any()
            is_present_meds = registration_form_data_export[(registration_form_data_export['participant_meds'] != "no")
                                                       & (registration_form_data_export['recentmeds_desc'] != "")]['recentmeds_desc'].str.contains(med, case=False, regex=True).any()
            other_ispresent_meds=registration_form_data_export[(registration_form_data_export['participant_psyccond'] != "no")
                                                                & (registration_form_data_export['psyccond_desc'] != "")]['psyccond_desc'].str.contains(med, case=False, regex=True).any()      
            specific_psych=registration_form_data_export[(registration_form_data_export['participant_psycspec'] != "no")
                                                                & (registration_form_data_export['psycspec_desc'] != "")]['psycspec_desc'].str.contains(med, case=False, regex=True).any()      
                              
            if is_present:
                # Set the new column to the value of key where med is present
                registration_form_data_export.loc[registration_form_data_export['psycmeds_desc'].str.contains(med).fillna(False), med_group]=key
            if is_present_meds:
                registration_form_data_export.loc[registration_form_data_export['recentmeds_desc'].str.contains(med).fillna(False), med_group]=key
            if other_ispresent_meds:
                registration_form_data_export.loc[registration_form_data_export['psyccond_desc'].str.contains(med).fillna(False), med_group]=key
            if specific_psych:
                registration_form_data_export.loc[registration_form_data_export['psycspec_desc'].str.contains(med).fillna(False), med_group]=key

In [ ]:
# display(registration_form_data_export)

In [ ]:
for key in diagnosis1.keys():
    for dx_group in diagnosis1[key]:
        for i, dx in enumerate(diagnosis1[key][dx_group]):
            # Check if the med value is present in the psycmeds_desc column
           
            is_present=registration_form_data_export[(registration_form_data_export['participant_psyccond'] != "no")
                                                                & (registration_form_data_export['psyccond_desc'] != "")]['psyccond_desc'].str.contains(dx, case=False, regex=True).any()
            anxiety_dep_sz_specific=registration_form_data_export[(registration_form_data_export['participant_psycspec'] != "no")
                                                     & (registration_form_data_export['psycspec_desc'] != "")]['psycspec_desc'].str.contains(dx, case=False, regex=True).any()                                         
            if is_present:
                # Set the new column to the value of key where med is present
                registration_form_data_export.loc[registration_form_data_export['psyccond_desc'].str.contains(dx).fillna(False), dx_group]=key
            if anxiety_dep_sz_specific:
                registration_form_data_export.loc[registration_form_data_export['psycspec_desc'].str.contains(dx).fillna(False), dx_group]=key
            else:
                registration_form_data_export.loc[registration_form_data_export['psycspec_desc'].str.contains(dx).fillna(False), 'other']='1'
                
# display(registration_form_data_export)

In [ ]:

registration_form_data_export.rename(columns={'ADHD/ADD':'psyc_cond_checkbox___1',
'Aggression':'psyc_cond_checkbox___2',
'Anxiety (Generalized, Separation, Social, Panic, Attachment)':'psyc_cond_checkbox___3',
'ASD/Autism':'psyc_cond_checkbox___4',
'Aspergers':'psyc_cond_checkbox___5',
'Bipolar disorder':'psyc_cond_checkbox___6',
'Depression/Major Depression Disorder (MDD)':'psyc_cond_checkbox___7',
'Emotional Disturbance Disorder/Other Mood Disorder':'psyc_cond_checkbox___8',
'Obssessive Compulsive Disorder (OCD)':'psyc_cond_checkbox___9',
'Oppositional Defiant Disorder (ODD)':'psyc_cond_checkbox___10',
'Pervasive Developmental Disorder Not Otherwise Specified (PDD-NOS)':'psyc_cond_checkbox___11',
'Phobia/specific fear (e.g., claustrophobia)':'psyc_cond_checkbox___12',
'PTSD/Trauma/Adjustment Disorder':'psyc_cond_checkbox___13',
'Selective Mutism':'psyc_cond_checkbox___14',
'Sensory Integration/Processing Disorder':'psyc_cond_checkbox___15',
'Tic/Tourettes':'psyc_cond_checkbox___16',
'Trichiyillomania':'psyc_cond_checkbox___17',
'other':'psyc_cond_checkbox___99',
'NA (not psychological; not listed above)':'psyc_cond_checkbox___98'}).to_csv(output_filenamne_prefix+'psych_fixed_and_specific_psych.csv')